In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader
from torchvision.models import efficientnet_v2_s

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28,28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)
def imshow(img):
         img = img / 2 + 0.5  # unnormalize
         npimg = img.numpy()
         npimg=np.clip(npimg, 0, 10)
         plt.imshow(np.transpose(npimg, (1, 2, 0)))
         plt.show()
     # Get some random training images
dataiter = iter(train_loader)
images, labels = next(dataiter)
     # Show images
imshow(torchvision.utils.make_grid(images[:4]))
     # Print labels
print(' '.join(f'{letters[labels[j]]:5s}' for j in range(4)))



In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here
class efficientnet_v2_s(nn.Module):
         def __init__(self, num_classes=10):
             super(efficientnet_v2_s, self).__init__()
model = torchvision.models.efficientnet_v2_s(pretrained=True)
 # Freeze all parameters in the feature extractor


model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [ ]:
model = torchvision.models.efficientnet_v2_s(pretrained=True)
print(model)

In [ ]:
# Write your code here

def train_one_epoch(model, dataloader, criterion, optimizer, device):
     model.train()
     total_loss = 0
     correct = 0

     total = 0
     for images, labels in tqdm(dataloader):
         images, labels = images.to(device), labels.to(device)
         labels=labels-1
         outputs = model(images).squeeze()  # The model outputs in shap
         loss = criterion(outputs, labels)
         optimizer.zero_grad()
         loss.backward()
         optimizer.step()
         total_loss += loss.item()
    # Track accuracy
         predictions = (outputs).argmax(dim=1)  # Get predicted c

         correct += (predictions == labels).sum().item()
         total += labels.size(0)
     avg_loss = total_loss / len(dataloader)
     accuracy = 100 * correct / total
     return avg_loss, accuracy

#validate

def validate(model, dataloader, criterion, device):
     model.eval()
     total_loss = 0
     correct = 0
     total = 0
     with torch.no_grad():
         for images, labels in dataloader:
             images, labels = images.to(device), labels.to(device)
             labels=labels-1
             outputs = model(images).squeeze()  # The model outputs in
             loss = criterion(outputs, labels)
             total_loss += loss.item()
             # Compute accuracy
             predictions = (outputs).argmax(dim=1) # Get predicte
             correct += (predictions == labels).sum().item()
             total += labels.size(0)
     avg_loss = total_loss / len(dataloader)
     accuracy = 100 * correct / total
     return avg_loss, accuracy


In [ ]:
# Write your code here
import torch
from torch import nn
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)
num_epochs = 5  # Define number of epochs
train_losses = []
val_losses = []
# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

In [ ]:
# Write your code here
